# 🔥 Natural Gas Price Forecasting
**Prepared by:** Ojas Khetarpal — Summer Trainee

This notebook:
1. Fetches historical Henry Hub gas prices from the EIA API
2. Explores and visualises the data
3. Trains a Facebook Prophet forecasting model
4. Generates a 90-day price forecast
5. Saves the model and forecast for use in the Streamlit dashboard

## 📦 Step 1 — Install dependencies

In [1]:
!pip install prophet plotly requests --quiet
print('✅ Done installing')

✅ Done installing



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔑 Step 2 — Set your EIA API key

In [ ]:
EIA_API_KEY = 'VffCe41jW7Qle7DZzVgcP0HLjxK2aPYTTc234Cyb'


EIA_BASE_URL = 'https://api.eia.gov/v2'
FORECAST_DAYS = 90

print('✅ API key set')

✅ API key set


## 📡 Step 3 — Fetch data from EIA API

We're pulling **Henry Hub Natural Gas Spot Price** (daily, $/MMBtu) — the benchmark price for natural gas in the US.

In [3]:
import requests
import pandas as pd

def fetch_eia_gas_prices(api_key, start='2015-01-01'):
    url = f"{EIA_BASE_URL}/natural-gas/pri/fut/data/"
    params = {
        'api_key': api_key,
        'frequency': 'daily',
        'data[0]': 'value',
        'facets[series][]': 'RNGWHHD',
        'start': start,
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
        'offset': 0,
        'length': 5000
    }

    all_data = []
    while True:
        response = requests.get(url, params=params)
        response.raise_for_status()
        result = response.json()
        data = result.get('response', {}).get('data', [])
        if not data:
            break
        all_data.extend(data)
        if len(data) < 5000:
            break
        params['offset'] += 5000

    df = pd.DataFrame(all_data)
    df = df[['period', 'value']].copy()
    df.columns = ['date', 'price']
    df['date'] = pd.to_datetime(df['date'])
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    df = df.dropna().sort_values('date').reset_index(drop=True)
    return df

df_raw = fetch_eia_gas_prices(EIA_API_KEY)
print(f'✅ Fetched {len(df_raw):,} records')
print(f'   Date range: {df_raw.date.min().date()} → {df_raw.date.max().date()}')
df_raw.tail(5)

✅ Fetched 2,881 records
   Date range: 2015-01-02 → 2026-06-01


,date,price
2876,2026-05-26,3.10
2877,2026-05-27,3.13
2878,2026-05-28,3.04
2879,2026-05-29,3.34
2880,2026-06-01,3.07


## 🔍 Step 4 — Explore the data

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# Basic stats
print('=== Summary Statistics ===')
print(df_raw['price'].describe().round(2))
print(f'\nMissing values: {df_raw.isnull().sum().sum()}')

=== Summary Statistics ===
count    2881.00
mean        3.16
std         1.61
min         1.21
25%         2.40
50%         2.82
75%         3.22
max        30.72
Name: price, dtype: float64

Missing values: 0


In [ ]:
# Full price history chart

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_raw['date'],
    y=df_raw['price'],
    mode='lines',
    name='Henry Hub Price',
    line=dict(color='#2E75B6', width=1.2)
))
fig.update_layout(
    title='Henry Hub Natural Gas Spot Price ($/MMBtu)',
    xaxis_title='Date',
    yaxis_title='Price ($/MMBtu)',
    hovermode='x unified',
    height=400
)
fig.show()

In [ ]:
# Yearly average prices
df_raw['year'] = df_raw['date'].dt.year
yearly = df_raw.groupby('year')['price'].mean().reset_index()

fig2 = px.bar(
    yearly,
    x='year', y='price',
    title='Average Annual Gas Price ($/MMBtu)',
    color='price',
    color_continuous_scale='Blues',
    labels={'price': 'Avg Price ($/MMBtu)', 'year': 'Year'}
)
fig2.update_layout(height=380, coloraxis_showscale=False)
fig2.show()

In [ ]:
# Seasonal pattern — average price by month
df_raw['month'] = df_raw['date'].dt.month
monthly = df_raw.groupby('month')['price'].mean().reset_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['month_name'] = monthly['month'].apply(lambda x: month_names[x-1])

fig3 = px.line(
    monthly, x='month_name', y='price',
    title='Average Price by Month (Seasonal Pattern)',
    markers=True,
    labels={'price': 'Avg Price ($/MMBtu)', 'month_name': 'Month'}
)
fig3.update_traces(line_color='#2E75B6')
fig3.update_layout(height=350)
fig3.show()

## 🧹 Step 5 — Clean & prepare data for Prophet

Prophet requires exactly two columns: `ds` (date) and `y` (value).

In [ ]:
import numpy as np

# Prepare for Prophet
df_prophet = df_raw[['date', 'price']].copy()
df_prophet.columns = ['ds', 'y']

# Remove outliers beyond 3 standard deviations
mean, std = df_prophet['y'].mean(), df_prophet['y'].std()
before = len(df_prophet)
df_prophet = df_prophet[np.abs(df_prophet['y'] - mean) < 3 * std].copy()
print(f'Removed {before - len(df_prophet)} outliers')

# Fill any gaps in dates with forward fill
df_prophet = df_prophet.set_index('ds').resample('D').ffill().reset_index()

print(f'✅ Data ready: {len(df_prophet):,} rows')
df_prophet.tail(3)

## 🤖 Step 6 — Train the Prophet model

In [ ]:
from prophet import Prophet

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='multiplicative',  # good for commodity prices
    changepoint_prior_scale=0.1,        # controls trend flexibility
    interval_width=0.95                 # 95% confidence interval
)

print('Training model...')
model.fit(df_prophet)
print('✅ Model trained')

## 📈 Step 7 — Generate forecast

In [ ]:
# Create future dataframe
future = model.make_future_dataframe(periods=FORECAST_DAYS, freq='D')
forecast = model.predict(future)

print(f'✅ Forecast generated: {FORECAST_DAYS} days ahead')
print(f'   Predicted price range: ${forecast["yhat"].tail(FORECAST_DAYS).min():.2f} — ${forecast["yhat"].tail(FORECAST_DAYS).max():.2f}/MMBtu')
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

In [ ]:
# Plot forecast with confidence intervals
hist = df_prophet.tail(365)  # show last year of history
pred = forecast.tail(FORECAST_DAYS)

fig4 = go.Figure()

# Confidence interval band
fig4.add_trace(go.Scatter(
    x=pd.concat([pred['ds'], pred['ds'][::-1]]),
    y=pd.concat([pred['yhat_upper'], pred['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(46, 117, 182, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Confidence Interval'
))

# Historical prices
fig4.add_trace(go.Scatter(
    x=hist['ds'], y=hist['y'],
    mode='lines', name='Historical Price',
    line=dict(color='#1F4E79', width=1.5)
))

# Forecast line
fig4.add_trace(go.Scatter(
    x=pred['ds'], y=pred['yhat'],
    mode='lines', name='Forecast',
    line=dict(color='#E74C3C', width=2, dash='dash')
))

fig4.update_layout(
    title=f'Natural Gas Price Forecast — Next {FORECAST_DAYS} Days',
    xaxis_title='Date',
    yaxis_title='Price ($/MMBtu)',
    hovermode='x unified',
    height=450,
    legend=dict(x=0.01, y=0.99)
)
fig4.show()

In [ ]:
# Prophet's built-in component plots (trend + seasonality breakdown)
fig5 = model.plot_components(forecast)
fig5.suptitle('Forecast Components: Trend & Seasonality', fontsize=13)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## 📊 Step 8 — Model evaluation

We use cross-validation to measure how accurate the model is on past data.

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

print('Running cross-validation (this takes ~1 min)...')
df_cv = cross_validation(
    model,
    initial='730 days',   # train on first 2 years
    period='90 days',     # retrain every 90 days
    horizon='90 days'     # evaluate on next 90 days
)

df_perf = performance_metrics(df_cv)
print('\n✅ Cross-validation complete')
print(f"   Mean Absolute Error (MAE):  ${df_perf['mae'].mean():.3f}/MMBtu")
print(f"   Root Mean Sq Error (RMSE): ${df_perf['rmse'].mean():.3f}/MMBtu")
print(f"   MAPE:                       {df_perf['mape'].mean()*100:.1f}%")
df_perf[['horizon', 'mae', 'rmse', 'mape']].head(10)

## 💾 Step 9 — Save model and forecast outputs

In [ ]:
import pickle
import os

os.makedirs('outputs', exist_ok=True)

# Save trained model
with open('outputs/prophet_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('✅ Model saved → outputs/prophet_model.pkl')

# Save raw data
df_raw.to_csv('outputs/raw_prices.csv', index=False)
print('✅ Raw data saved → outputs/raw_prices.csv')

# Save forecast
forecast_out = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_out.columns = ['date', 'forecast', 'lower', 'upper']
forecast_out.to_csv('outputs/forecast.csv', index=False)
print('✅ Forecast saved → outputs/forecast.csv')

# Download files if running in Google Colab
try:
    from google.colab import files
    print('\nDownloading files to your computer...')
    files.download('outputs/prophet_model.pkl')
    files.download('outputs/raw_prices.csv')
    files.download('outputs/forecast.csv')
except ImportError:
    print('\nRunning locally; outputs saved to standard folders.')

print('✅ All done! Drop these files into your local project:')
print('   prophet_model.pkl → gas_forecast/models/')
print('   raw_prices.csv    → gas_forecast/data/')
print('   forecast.csv      → gas_forecast/data/')